In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Function to generate a random discrete signal
def generate_random_signal(k):
    n = 2 ** k
    signal = np.random.randn(n)
    return signal

# DFT Implementation
def dft(signal):
    N = len(signal)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-2j * np.pi * k * n / N)
    return np.dot(W, signal)

# IDFT Implementation
def idft(frequency_domain_signal):
    N = len(frequency_domain_signal)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W_inv = np.exp(2j * np.pi * k * n / N) / N
    return np.dot(W_inv, frequency_domain_signal)

# Fast Fourier Transform (FFT)
def fft(P):
    n = len(P)
    if n <= 1:
        return P  

    # Recursive calls
    P_even = fft(P[::2])
    P_odd = fft(P[1::2])

    # Combine
    y = np.zeros(n, dtype=complex)
    for i in range(n // 2):
        factor = np.exp(-2j * np.pi * i / n) * P_odd[i]
        y[i] = P_even[i] + factor
        y[i + n // 2] = P_even[i] - factor
    return y

# Inverse Fast Fourier Transform (IFFT)
def ifft(P):
    n = len(P)
    if n <= 1:
        return P  

    # Recursive calls
    P_even = ifft(P[::2])
    P_odd = ifft(P[1::2])

    # Combine
    y = np.zeros(n, dtype=complex)
    for i in range(n // 2):
        factor = np.exp(2j * np.pi * i / n) * P_odd[i]
        y[i] = P_even[i] + factor
        y[i + n // 2] = P_even[i] - factor

    # Normalize at the final step
    return y / n


# Function to compare runtimes of DFT, FFT, IDFT, and IFFT
def compare_runtimes(max_k=10, trials=10):
    dft_times = []
    fft_times = []
    idft_times = []
    ifft_times = []
    sizes = []

    for k in range(2, max_k + 1):
        N = 2 ** k
        sizes.append(N)

        dft_avg_time = 0
        fft_avg_time = 0
        idft_avg_time = 0
        ifft_avg_time = 0

        for _ in range(trials):
            signal = generate_random_signal(k)

            # Measure DFT Time
            start = time.perf_counter()
            dft_signal = dft(signal)
            dft_avg_time += (time.perf_counter() - start)

            # Measure FFT Time
            start = time.perf_counter()
            fft_signal = np.fft.fftn(signal)
            fft_avg_time += (time.perf_counter() - start)

            # Measure IDFT Time
            start = time.perf_counter()
            idft_signal = idft(dft_signal)
            idft_avg_time += (time.perf_counter() - start)

            # Measure IFFT Time
            start = time.perf_counter()
            ifft_signal = np.fft.ifft(fft_signal)
            ifft_avg_time += (time.perf_counter() - start)

        dft_times.append(dft_avg_time / trials)
        fft_times.append(fft_avg_time / trials)
        idft_times.append(idft_avg_time / trials)
        ifft_times.append(ifft_avg_time / trials)

    # Plotting DFT vs FFT runtime
    plt.figure(figsize=(12, 6))
    plt.plot(sizes, dft_times, label='DFT', marker='o', color='blue', linestyle='-')
    plt.plot(sizes, fft_times, label='FFT', marker='o', color='red', linestyle='--')

    plt.xscale('log', base=2)
    plt.yscale('log')
    plt.xlabel('Signal Size (n)')
    plt.ylabel('Runtime (seconds)')
    plt.title('DFT vs FFT Runtime Comparison')
    plt.legend()
    plt.grid(True)
    plt.savefig("DFT_Vs_FFT.png")
    plt.clf()

    # Plotting IDFT vs IFFT runtime
    plt.figure(figsize=(12, 6))
    plt.plot(sizes, idft_times, label='IDFT', marker='o', color='blue', linestyle='-')
    plt.plot(sizes, ifft_times, label='IFFT', marker='o', color='red', linestyle='--')

    plt.xscale('log', base=2)
    plt.yscale('log')
    plt.xlabel('Signal Size (n)')
    plt.ylabel('Runtime (seconds)')
    plt.title('IDFT vs IFFT Runtime Comparison')
    plt.legend()
    plt.grid(True)
    plt.savefig("IDFT_Vs_IFFT.png")
    plt.clf()

# Run the comparison
compare_runtimes()